In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

class SILRHunter:
    def __init__(self):
        self.results = {}
        
    def load_telemetry(self, filepath: str) -> pd.DataFrame:
        """Load telemetry data with required columns"""
        df = pd.read_csv(filepath)
        required_cols = ['timestamp', 'estimate', 'uncertainty', 'action', 'target']
        assert all(col in df.columns for col in required_cols), "Missing required columns"
        return df
    
    def compute_z_distribution(self, df: pd.DataFrame) -> Dict:
        """Compute z-score distribution and compare to half-normal"""
        z_scores = np.abs(df['estimate'] - df['target']) / df['uncertainty']
        
        # KS test against half-normal
        ks_stat, p_value = stats.kstest(z_scores, 'halfnorm')
        
        # Fit half-normal
        scale = np.sqrt(np.pi / 2) * np.mean(z_scores)
        fitted_halfnorm = stats.halfnorm(scale=scale)
        
        # QQ plot data
        n = len(z_scores)
        theoretical_quantiles = fitted_halfnorm.ppf(np.linspace(0, 1, n))
        
        return {
            'z_scores': z_scores,
            'ks_statistic': ks_stat,
            'p_value': p_value,
            'scale_parameter': scale,
            'theoretical_quantiles': theoretical_quantiles
        }
    
    def estimate_controller_params(self, df: pd.DataFrame) -> Tuple[float, float]:
        """Estimate B and z0 from data using MLE"""
        # Simplified estimation - in practice would use more sophisticated method
        z_scores = np.abs(df['estimate'] - df['target']) / df['uncertainty']
        leak_events = (df['action'] > df['action'].quantile(0.5)).astype(float)
        
        # Grid search for parameters that maximize likelihood
        best_ll = -np.inf
        best_B, best_z0 = 0.1, 1.0
        
        B_vals = np.linspace(0.1, 10, 50)
        z0_vals = np.linspace(0.1, 5, 50)
        
        for B in B_vals:
            for z0 in z0_vals:
                probs = 1 / (1 + np.exp(-B * (z_scores - z0)))
                ll = np.sum(leak_events * np.log(probs + 1e-10) + 
                           (1 - leak_events) * np.log(1 - probs + 1e-10))
                if ll > best_ll:
                    best_ll = ll
                    best_B, best_z0 = B, z0
        
        return best_B, best_z0
    
    def test_scale_invariance(self, df: pd.DataFrame, B: float, z0: float, 
                             n_scales: int = 20, n_bootstrap: int = 100) -> Dict:
        """Main SILR test: does leak rate stay constant when scaling uncertainty?"""
        scale_factors = np.logspace(-1, 1, n_scales)  # 0.1 to 10
        
        baseline_z = np.abs(df['estimate'] - df['target']) / df['uncertainty']
        baseline_probs = 1 / (1 + np.exp(-B * (baseline_z - z0)))
        baseline_leak_rate = np.mean(baseline_probs > np.random.random(len(baseline_probs)))
        
        scale_results = []
        
        for scale in scale_factors:
            scaled_z = baseline_z / scale
            scaled_probs = 1 / (1 + np.exp(-B * (scaled_z - z0)))
            
            # Bootstrap to get confidence intervals
            boot_leak_rates = []
            for _ in range(n_bootstrap):
                idx = np.random.choice(len(scaled_probs), len(scaled_probs), replace=True)
                sample_leaks = scaled_probs[idx] > np.random.random(len(idx))
                boot_leak_rates.append(np.mean(sample_leaks))
            
            scale_results.append({
                'scale': scale,
                'mean_leak_rate': np.mean(boot_leak_rates),
                'ci_low': np.percentile(boot_leak_rates, 2.5),
                'ci_high': np.percentile(boot_leak_rates, 97.5),
                'std': np.std(boot_leak_rates)
            })
        
        # Compute invariance metric
        leak_rates = [r['mean_leak_rate'] for r in scale_results]
        max_deviation = np.max(np.abs(np.array(leak_rates) - baseline_leak_rate))
        relative_deviation = max_deviation / baseline_leak_rate if baseline_leak_rate > 0 else np.inf
        
        is_invariant = relative_deviation < 0.1  # Less than 10% deviation
        
        return {
            'scale_factors': scale_factors,
            'results': scale_results,
            'baseline_leak_rate': baseline_leak_rate,
            'max_deviation': max_deviation,
            'relative_deviation': relative_deviation,
            'is_invariant': is_invariant,
            'invariant_strength': 1.0 / (relative_deviation + 1e-10)
        }
    
    def analyze_system(self, telemetry_path: str) -> Dict:
        """Complete analysis pipeline"""
        print(f"🔍 Analyzing system: {telemetry_path}")
        
        # Load data
        df = self.load_telemetry(telemetry_path)
        print(f"   Loaded {len(df)} data points")
        
        # Step 1: Z-distribution analysis
        print("   Step 1: Analyzing z-score distribution...")
        z_analysis = self.compute_z_distribution(df)
        
        # Step 2: Estimate controller parameters
        print("   Step 2: Estimating controller parameters...")
        B, z0 = self.estimate_controller_params(df)
        print(f"     Estimated B={B:.3f}, z0={z0:.3f}")
        
        # Step 3: Scale invariance test
        print("   Step 3: Testing scale invariance...")
        invariance_test = self.test_scale_invariance(df, B, z0)
        
        # Compile results
        self.results = {
            'metadata': {
                'data_points': len(df),
                'estimated_B': B,
                'estimated_z0': z0,
                'z_distribution_ks': z_analysis['ks_statistic'],
                'z_distribution_p': z_analysis['p_value']
            },
            'invariance_test': invariance_test,
            'z_analysis': z_analysis
        }
        
        return self.results
    
    def generate_report(self, output_path: str = "silr_report.html"):
        """Generate HTML report with visualizations"""
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
        
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('Z-Score Distribution vs Half-Normal',
                          'Scale Invariance Test',
                          'QQ Plot: Empirical vs Theoretical',
                          'Leak Rate vs Uncertainty Scaling'),
            vertical_spacing=0.15,
            horizontal_spacing=0.15
        )
        
        # 1. Z-score histogram vs half-normal
        z_scores = self.results['z_analysis']['z_scores']
        scale = self.results['z_analysis']['scale_parameter']
        x_range = np.linspace(0, np.max(z_scores) * 1.1, 100)
        pdf = stats.halfnorm.pdf(x_range, scale=scale)
        
        fig.add_trace(
            go.Histogram(x=z_scores, nbinsx=50, name='Empirical', opacity=0.7),
            row=1, col=1
        )
        fig.add_trace(
            go.Scatter(x=x_range, y=pdf * len(z_scores) * (x_range[1]-x_range[0]), 
                      name='Half-Normal Fit', line=dict(color='red')),
            row=1, col=1
        )
        
        # 2. Scale invariance test
        test_results = self.results['invariance_test']['results']
        scales = [r['scale'] for r in test_results]
        leak_rates = [r['mean_leak_rate'] for r in test_results]
        ci_low = [r['ci_low'] for r in test_results]
        ci_high = [r['ci_high'] for r in test_results]
        
        fig.add_trace(
            go.Scatter(x=scales, y=leak_rates, mode='lines+markers', 
                      name='Leak Rate', line=dict(color='blue')),
            row=1, col=2
        )
        fig.add_trace(
            go.Scatter(x=scales, y=ci_low, mode='lines', 
                      name='95% CI', line=dict(color='lightblue', dash='dash'),
                      showlegend=False),
            row=1, col=2
        )
        fig.add_trace(
            go.Scatter(x=scales, y=ci_high, mode='lines', 
                      line=dict(color='lightblue', dash='dash'),
                      fill='tonexty', fillcolor='rgba(173,216,230,0.3)',
                      name='95% CI'),
            row=1, col=2
        )
        
        # Add horizontal line for baseline
        baseline = self.results['invariance_test']['baseline_leak_rate']
        fig.add_hline(y=baseline, line_dash="dot", row=1, col=2,
                     annotation_text=f"Baseline: {baseline:.3f}")
        
        # 3. QQ plot
        theoretical = self.results['z_analysis']['theoretical_quantiles']
        empirical = np.sort(z_scores)
        
        fig.add_trace(
            go.Scatter(x=theoretical, y=empirical, mode='markers',
                      marker=dict(size=4), name='QQ Plot'),
            row=2, col=1
        )
        fig.add_trace(
            go.Scatter(x=[0, max(theoretical)], y=[0, max(theoretical)],
                      mode='lines', line=dict(color='red', dash='dash'),
                      name='y=x'),
            row=2, col=1
        )
        
        # 4. Deviation plot
        deviations = [abs(lr - baseline) for lr in leak_rates]
        fig.add_trace(
            go.Scatter(x=scales, y=deviations, mode='lines+markers',
                      name='Deviation from Baseline', line=dict(color='orange')),
            row=2, col=2
        )
        
        # Update layout
        fig.update_layout(height=800, showlegend=True, 
                         title_text=f"SILR Analysis Report - Invariant: {self.results['invariance_test']['is_invariant']}")
        
        # Save report
        fig.write_html(output_path)
        print(f"📊 Report saved to {output_path}")
        
        return output_path
    
    def get_verdict(self) -> str:
        """Generate plain English verdict"""
        results = self.results
        
        if results['invariance_test']['is_invariant']:
            strength = results['invariance_test']['invariant_strength']
            deviation = results['invariance_test']['relative_deviation']
            
            if deviation < 0.05:
                verdict = "STRONG SILR DETECTED"
                confidence = "High"
            elif deviation < 0.1:
                verdict = "MODERATE SILR DETECTED"
                confidence = "Medium"
            else:
                verdict = "WEAK SILR DETECTED"
                confidence = "Low"
                
            return f"""
            ✅ {verdict}
            
            Key Evidence:
            • Scale invariance: Leak rate varies only {deviation*100:.1f}% across 10x uncertainty scaling
            • Z-distribution: KS statistic = {results['metadata']['z_distribution_ks']:.3f} (p={results['metadata']['z_distribution_p']:.3f})
            • Controller parameters: B={results['metadata']['estimated_B']:.2f}, z0={results['metadata']['estimated_z0']:.2f}
            
            Confidence: {confidence}
            
            This system exhibits scale-invariant leakage behavior consistent with SILR theory.
            """
        else:
            return f"""
            ❌ NO SILR DETECTED
            
            Key Evidence:
            • Scale invariance FAILED: Leak rate varies {results['invariance_test']['relative_deviation']*100:.1f}% across uncertainty scaling
            • This system does NOT exhibit scale-invariant leakage
            
            Possible reasons:
            1. Controller doesn't use z-score gating
            2. Uncertainty estimates are miscalibrated
            3. Leak mechanism differs from sigmoid gating
            4. System has additional nonlinearities
            """

# Example usage with simulated data
def generate_test_data(n=10000, B=5.0, z0=2.0, noise_scale=1.0):
    """Generate synthetic data that SHOULD exhibit SILR"""
    np.random.seed(42)
    
    # True state (random walk)
    true_state = np.cumsum(np.random.randn(n) * 0.1)
    
    # Estimates with noise
    estimates = true_state + np.random.randn(n) * noise_scale
    
    # Uncertainties (varying)
    uncertainties = noise_scale * (0.5 + 0.5 * np.random.rand(n))
    
    # Controller decisions
    z_scores = np.abs(estimates) / uncertainties  # target = 0
    probs = 1 / (1 + np.exp(-B * (z_scores - z0)))
    actions = (probs > np.random.rand(n)).astype(float)
    
    # Add some random actions (10% noise)
    noise_mask = np.random.rand(n) < 0.1
    actions[noise_mask] = 1 - actions[noise_mask]
    
    return pd.DataFrame({
        'timestamp': np.arange(n),
        'estimate': estimates,
        'uncertainty': uncertainties,
        'action': actions,
        'target': np.zeros(n)
    })

if __name__ == "__main__":
    print("="*60)
    print("SILR-HUNTER: Finding Scale-Invariant Leakage in the Wild")
    print("="*60)
    
    # Generate test data
    print("\n1️⃣ Generating test data...")
    test_df = generate_test_data()
    test_df.to_csv("test_telemetry.csv", index=False)
    print(f"   Generated {len(test_df)} data points")
    print(f"   Saved to test_telemetry.csv")
    
    # Run analysis
    print("\n2️⃣ Running SILR analysis...")
    hunter = SILRHunter()
    results = hunter.analyze_system("test_telemetry.csv")
    
    # Generate report
    print("\n3️⃣ Generating report...")
    report_path = hunter.generate_report("silr_analysis_report.html")
    
    # Print verdict
    print("\n4️⃣ VERDICT:")
    print(hunter.get_verdict())
    
    print("\n" + "="*60)
    print("Next steps:")
    print("1. Use this code on REAL telemetry data")
    print("2. Check if leak rate stays constant when scaling uncertainty")
    print("3. If invariant, you've found SILR in the wild!")
    print("="*60)

SILR-HUNTER: Finding Scale-Invariant Leakage in the Wild

1️⃣ Generating test data...
   Generated 10000 data points
   Saved to test_telemetry.csv

2️⃣ Running SILR analysis...
🔍 Analyzing system: test_telemetry.csv
   Loaded 10000 data points
   Step 1: Analyzing z-score distribution...
   Step 2: Estimating controller parameters...
     Estimated B=0.100, z0=5.000
   Step 3: Testing scale invariance...

3️⃣ Generating report...
📊 Report saved to silr_analysis_report.html

4️⃣ VERDICT:

            ❌ NO SILR DETECTED

            Key Evidence:
            • Scale invariance FAILED: Leak rate varies 70.9% across uncertainty scaling
            • This system does NOT exhibit scale-invariant leakage

            Possible reasons:
            1. Controller doesn't use z-score gating
            2. Uncertainty estimates are miscalibrated
            3. Leak mechanism differs from sigmoid gating
            4. System has additional nonlinearities
            

Next steps:
1. Use this code 